In [ ]:
!pip install tensorflow scikit-learn pandas numpy nltk matplotlib seaborn

import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
print("All imports done!")

In [ ]:
#load and combine dataset
from google.colab import files
uploaded = files.upload()  # upload all 3 CSV files

df_sexism = pd.read_csv('twitter_sexism_parsed_dataset.csv')
df_racism = pd.read_csv('twitter_racism_parsed_dataset.csv')
df_parsed  = pd.read_csv('twitter_parsed_dataset.csv')

# Combine into one dataframe
df = pd.concat([df_sexism, df_racism, df_parsed], ignore_index=True)
print("Total records:", len(df))
df.head()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
#handling null values
df.isna().any()
df.isna().sum()

In [ ]:
df = df.dropna(subset=['oh_label', 'Text'])
df['oh_label'] = df['oh_label'].astype(int)

print("Null values after cleaning:")
print(df.isna().sum())
print("\nNew DataFrame shape:", df.shape)
display(df.head())

In [ ]:
#handling duplicate values
df.nunique()
df['Text'].nunique()

In [ ]:
# Remove duplicates based on the original Text column
before_count = len(df)
df = df.drop_duplicates(subset=['Text'])
after_count = len(df)

print(f"Rows before removing duplicates: {before_count}")
print(f"Rows after removing duplicates: {after_count}")
print(f"Total duplicates removed: {before_count - after_count}")

# Re-verify unique counts
print("\nUnique values in cleaned Text column:", df['Text'].nunique())
display(df.head())

In [ ]:
label_map = {0.0: 'Not Cyberbullying', 1.0: 'Cyberbullying'}
df['label_name'] = df['oh_label'].map(label_map)

# Display the count of new labels
print(df['label_name'].value_counts())

# Plot the updated distribution
plt.figure(figsize=(8, 6))
sns.countplot(x='label_name', data=df, palette='viridis')
plt.title('Distribution of Cyberbullying vs Not Cyberbullying')
plt.xlabel('Category')
plt.ylabel('Number of Tweets')
plt.show()

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()                          # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'@\w+', '', text)                  # remove mentions
    text = re.sub(r'#', '', text)                  # remove hashtags
    text = re.sub(r'[^a-z\s]', '', text)             # remove punctuation/numbers
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['Text'].astype(str).apply(clean_text)
df[['Text', 'clean_text', 'oh_label']].head()

In [ ]:
from collections import Counter

bully_text = ' '.join(df[df['oh_label'] == 1]['clean_text'])
bully_words = bully_text.split()
top_bully = Counter(bully_words).most_common(20)
print("Top cyberbullying keywords:")
for word, count in top_bully:
    print(f"  {word}: {count}")

In [ ]:
# Create a bar chart for the top cyberbullying words
words, counts = zip(*top_bully)

plt.figure(figsize=(12, 8))
# Updated to use hue=y and legend=False to resolve the FutureWarning
sns.barplot(x=list(counts), y=list(words), hue=list(words), palette='magma', legend=False)
plt.title('Top 20 Keywords in Cyberbullying Tweets')
plt.xlabel('Frequency')
plt.ylabel('Words')
plt.show()

In [ ]:
print(df['oh_label'].value_counts())

In [ ]:
X = df['clean_text']
y = df['oh_label']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
class_weight = {0: class_weights_array[0], 1: class_weights_array[1]}

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)
print("Testing data:", X_test.shape)
print("Class weights:", class_weight)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 100
embedding_dim = 64

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

print("Padded training shape:", X_train_pad.shape)
print("Padded validation shape:", X_val_pad.shape)
print("Padded testing shape:", X_test_pad.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.regularizers import l2

def build_cnn_model():
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
        SpatialDropout1D(0.3),
        Conv1D(filters=64, kernel_size=5, activation='relu', kernel_regularizer=l2(1e-4)),
        GlobalMaxPooling1D(),
        Dropout(0.5),
        Dense(32, activation='relu', kernel_regularizer=l2(1e-4)),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model

In [ ]:
from tensorflow.keras.layers import LSTM

def build_lstm_model():
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
        LSTM(128),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model


In [ ]:
from tensorflow.keras.layers import Bidirectional

def build_bilstm_model():
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
        Bidirectional(LSTM(128)),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model


In [ ]:
import time
from tensorflow.keras.callbacks import EarlyStopping

epochs = 10
batch_size = 32

models = {
    "CNN": build_cnn_model(),
    "LSTM": build_lstm_model(),
    "BiLSTM": build_bilstm_model()
}

histories = {}
training_times = {}

for name, model in models.items():
    print(f"\nTraining {name} model...")

    start_time = time.time()

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping],
        class_weight=class_weight,
        verbose=1
    )

    end_time = time.time()

    histories[name] = history
    training_times[name] = end_time - start_time

    print(f"{name} training completed.")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

results = []

for name, model in models.items():
    print(f"\nEvaluating {name} model...")

    y_pred_prob = model.predict(X_test_pad)
    y_pred = (y_pred_prob > 0.5).astype("int32")

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Training Time": training_times[name]
    })

    print(f"\n{name} Classification Report:")
    print(classification_report(y_test, y_pred))


In [ ]:
results_df = pd.DataFrame(results)
results_df


In [ ]:
plt.figure(figsize=(10, 6))

metrics_df = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]]

metrics_df.plot(kind="bar", figsize=(10, 6))
plt.title("Performance Comparison of CNN, LSTM, and BiLSTM Models")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()


In [ ]:
for name, model in models.items():
    y_pred_prob = model.predict(X_test_pad)
    y_pred = (y_pred_prob > 0.5).astype("int32")

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.show()


### Investigating LSTM Model Predictions

Zero precision and recall for a class indicate that the model did not predict any instances of that class. Let's inspect the predictions of the LSTM model to confirm this.

In [ ]:
lstm_model = models['LSTM']

# Get predictions for the LSTM model
lstm_y_pred_prob = lstm_model.predict(X_test_pad)
lstm_y_pred = (lstm_y_pred_prob > 0.5).astype("int32")

# Display the unique predictions and their counts
unique_predictions, counts = np.unique(lstm_y_pred, return_counts=True)
print("Unique predictions by LSTM model:")
for pred, count in zip(unique_predictions, counts):
    print(f"  Class {pred}: {count} predictions")

if len(unique_predictions) == 1 and unique_predictions[0] == 0:
    print("\nAs observed, the LSTM model only predicts '0' (Not Cyberbullying). Therefore, it never predicts '1' (Cyberbullying), leading to zero precision and recall for that class.")


In [ ]:
df['oh_label'].value_counts()


In [ ]:
df['oh_label'].value_counts(normalize=True) * 100


CNN hyperparameter setting best filter

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

def build_cnn_model(
    filters=64,
    kernel_size=5,
    dropout_rate=0.5,
    learning_rate=0.001,
    embedding_dim=64,
    l2_reg=1e-4,
    spatial_dropout=0.3
):
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
        SpatialDropout1D(spatial_dropout),
        Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            activation='relu',
            kernel_regularizer=l2(l2_reg)
        ),
        GlobalMaxPooling1D(),
        Dropout(dropout_rate),
        Dense(32, activation='relu', kernel_regularizer=l2(l2_reg)),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=learning_rate),
        metrics=['accuracy']
    )

    return model

In [ ]:
def evaluate_cnn_model(model):
    y_pred_prob = model.predict(X_val_pad)
    y_pred = (y_pred_prob > 0.5).astype("int32")

    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    return accuracy, precision, recall, f1

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

filter_values = [64, 128, 256]

filter_results = []

for filters in filter_values:
    print(f"\nTesting CNN filters: {filters}")

    model = build_cnn_model(
        filters=filters,
        kernel_size=5,
        dropout_rate=0.5,
        learning_rate=0.001
    )

    history = model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=20,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )

    accuracy, precision, recall, f1 = evaluate_cnn_model(model)

    filter_results.append({
        "Filters": filters,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

filter_results_df = pd.DataFrame(filter_results)
display(filter_results_df)

best_filter = filter_results_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Filters"]

print("Best Filter:", best_filter)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

kernel_values = [3, 5, 7]

kernel_results = []

for kernel_size in kernel_values:
    print(f"\nTesting CNN kernel size: {kernel_size}")

    model = build_cnn_model(
        filters=int(best_filter),
        kernel_size=kernel_size,
        dropout_rate=0.5,
        learning_rate=0.001
    )

    history = model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=20,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )

    accuracy, precision, recall, f1 = evaluate_cnn_model(model)

    kernel_results.append({
        "Filters": int(best_filter),
        "Kernel Size": kernel_size,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

kernel_results_df = pd.DataFrame(kernel_results)
display(kernel_results_df)

best_kernel_size = kernel_results_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Kernel Size"]

print("Best Kernel Size:", best_kernel_size)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

dropout_values = [0.2, 0.3, 0.5]

dropout_results = []

for dropout_rate in dropout_values:
    print(f"\nTesting CNN dropout rate: {dropout_rate}")

    model = build_cnn_model(
        filters=int(best_filter),
        kernel_size=int(best_kernel_size),
        dropout_rate=dropout_rate,
        learning_rate=0.001
    )

    history = model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=20,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )

    accuracy, precision, recall, f1 = evaluate_cnn_model(model)

    dropout_results.append({
        "Filters": int(best_filter),
        "Kernel Size": int(best_kernel_size),
        "Dropout Rate": dropout_rate,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

dropout_results_df = pd.DataFrame(dropout_results)
display(dropout_results_df)

best_dropout_rate = dropout_results_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Dropout Rate"]

print("Best Dropout Rate:", best_dropout_rate)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

learning_rate_values = [0.01, 0.001, 0.0001, 0.0005]

learning_rate_results = []

for learning_rate in learning_rate_values:
    print(f"\nTesting CNN learning rate: {learning_rate}")

    model = build_cnn_model(
        filters=int(best_filter),
        kernel_size=int(best_kernel_size),
        dropout_rate=float(best_dropout_rate),
        learning_rate=learning_rate
    )

    history = model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )

    accuracy, precision, recall, f1 = evaluate_cnn_model(model)

    learning_rate_results.append({
        "Filters": int(best_filter),
        "Kernel Size": int(best_kernel_size),
        "Dropout Rate": float(best_dropout_rate),
        "Learning Rate": learning_rate,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

learning_rate_results_df = pd.DataFrame(learning_rate_results)
display(learning_rate_results_df)

best_learning_rate = learning_rate_results_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Learning Rate"]

print("Best Learning Rate:", best_learning_rate)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

epoch_values = [30, 50, 100]

epoch_results = []

for epochs in epoch_values:
    print(f"\nTesting CNN epochs: {epochs}")

    model = build_cnn_model(
        filters=int(best_filter),
        kernel_size=int(best_kernel_size),
        dropout_rate=float(best_dropout_rate),
        learning_rate=float(best_learning_rate)
    )

    history = model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=epochs,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )

    accuracy, precision, recall, f1 = evaluate_cnn_model(model)

    epoch_results.append({
        "Filters": int(best_filter),
        "Kernel Size": int(best_kernel_size),
        "Dropout Rate": float(best_dropout_rate),
        "Learning Rate": float(best_learning_rate),
        "Epochs": epochs,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

epoch_results_df = pd.DataFrame(epoch_results)
display(epoch_results_df)

best_epochs = epoch_results_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Epochs"]

print("Best Epochs:", best_epochs)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

batch_size_values = [32, 64, 128]

batch_size_results = []

for batch_size in batch_size_values:
    print(f"\nTesting CNN batch size: {batch_size}")

    model = build_cnn_model(
        filters=int(best_filter),
        kernel_size=int(best_kernel_size),
        dropout_rate=float(best_dropout_rate),
        learning_rate=float(best_learning_rate)
    )

    history = model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=int(best_epochs),
        batch_size=batch_size,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )

    accuracy, precision, recall, f1 = evaluate_cnn_model(model)

    batch_size_results.append({
        "Filters": int(best_filter),
        "Kernel Size": int(best_kernel_size),
        "Dropout Rate": float(best_dropout_rate),
        "Learning Rate": float(best_learning_rate),
        "Epochs": int(best_epochs),
        "Batch Size": batch_size,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

batch_size_results_df = pd.DataFrame(batch_size_results)
display(batch_size_results_df)

best_batch_size = batch_size_results_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Batch Size"]

print("Best Batch Size:", best_batch_size)


In [ ]:
best_cnn_params = {
    "Best Filter": 64,
    "Best Kernel Size": 5,
    "Best Dropout Rate": 0.5,
    "Best Learning Rate": 0.001,
    "Best Epochs": 20,
    "Best Batch Size": 32
}

best_cnn_params

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

final_cnn_model = build_cnn_model(
    filters=best_cnn_params["Best Filter"],
    kernel_size=best_cnn_params["Best Kernel Size"],
    dropout_rate=best_cnn_params["Best Dropout Rate"],
    learning_rate=best_cnn_params["Best Learning Rate"]
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    min_lr=1e-5
)

final_cnn_history = final_cnn_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=best_cnn_params["Best Epochs"],
    batch_size=best_cnn_params["Best Batch Size"],
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weight,
    verbose=1
)

In [ ]:
y_pred_prob = final_cnn_model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype("int32")

print("Final CNN Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

final_cnn_accuracy = accuracy_score(y_test, y_pred)
final_cnn_precision = precision_score(y_test, y_pred, zero_division=0)
final_cnn_recall = recall_score(y_test, y_pred, zero_division=0)
final_cnn_f1 = f1_score(y_test, y_pred, zero_division=0)

final_cnn_results = pd.DataFrame([{
    "Model": "CNN",
    "Filters": best_cnn_params["Best Filter"],
    "Kernel Size": best_cnn_params["Best Kernel Size"],
    "Dropout Rate": best_cnn_params["Best Dropout Rate"],
    "Learning Rate": best_cnn_params["Best Learning Rate"],
    "Epochs": best_cnn_params["Best Epochs"],
    "Batch Size": best_cnn_params["Best Batch Size"],
    "Accuracy": final_cnn_accuracy,
    "Precision": final_cnn_precision,
    "Recall": final_cnn_recall,
    "F1-Score": final_cnn_f1
}])

display(final_cnn_results)


In [ ]:
all_cnn_tuning_results = pd.concat([
    filter_results_df.assign(Tuning_Stage="Filter"),
    kernel_results_df.assign(Tuning_Stage="Kernel Size"),
    dropout_results_df.assign(Tuning_Stage="Dropout Rate"),
    learning_rate_results_df.assign(Tuning_Stage="Learning Rate"),
    epoch_results_df.assign(Tuning_Stage="Epochs"),
    batch_size_results_df.assign(Tuning_Stage="Batch Size")
], ignore_index=True)

display(all_cnn_tuning_results)


### Training History of Final CNN Model: Accuracy and Loss over Epochs

In [ ]:
plt.figure(figsize=(14, 6))

# Plot Training & Validation Accuracy
plt.subplot(1, 2, 1)
plt.plot(final_cnn_history.history['accuracy'], label='Training Accuracy')
plt.plot(final_cnn_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Final CNN Model Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot Training & Validation Loss
plt.subplot(1, 2, 2)
plt.plot(final_cnn_history.history['loss'], label='Training Loss')
plt.plot(final_cnn_history.history['val_loss'], label='Validation Loss')
plt.title('Final CNN Model Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Baseline CNN Performance: Accuracy and Loss Graphs

In [ ]:
plt.figure(figsize=(14, 6))

# Get CNN history from the initial comparison
cnn_history = histories['CNN']

# Plot Training & Validation Accuracy
plt.subplot(1, 2, 1)
plt.plot(cnn_history.history['accuracy'], label='Training Accuracy')
plt.plot(cnn_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Baseline CNN Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot Training & Validation Loss
plt.subplot(1, 2, 2)
plt.plot(cnn_history.history['loss'], label='Training Loss')
plt.plot(cnn_history.history['val_loss'], label='Validation Loss')
plt.title('Baseline CNN Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### LSTM Hyperparameter Tuning: Units

In [ ]:
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.regularizers import l2

def build_lstm_model_tuned(units=64, dropout_rate=0.3, learning_rate=0.001, embedding_dim=64):
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
        SpatialDropout1D(0.3),
        LSTM(units, dropout=dropout_rate),
        Dense(32, activation='relu', kernel_regularizer=l2(1e-4)),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=learning_rate),
        metrics=['accuracy']
    )
    return model

In [ ]:
def evaluate_lstm_model(model):
    y_pred_prob = model.predict(X_val_pad)
    y_pred = (y_pred_prob > 0.5).astype("int32")

    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    return accuracy, precision, recall, f1

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

unit_values = [64, 128, 256]

units_results = []

for units in unit_values:
    print(f"\nTesting LSTM units: {units}")

    model = build_lstm_model_tuned(
        units=units,
        dropout_rate=0.5,
        learning_rate=0.001
    )

    history = model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=10, # Limiting epochs for initial tuning
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )

    accuracy, precision, recall, f1 = evaluate_lstm_model(model)

    units_results.append({
        "LSTM Units": units,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

units_results_df = pd.DataFrame(units_results)
display(units_results_df)

best_lstm_units = units_results_df.sort_values(by="F1-Score", ascending=False).iloc[0]["LSTM Units"]

print("Best LSTM Units:", best_lstm_units)

### LSTM Hyperparameter Tuning: Dropout Rate

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

lstm_dropout_values = [0.2, 0.3, 0.5]
lstm_dropout_results = []
for lstm_dropout in lstm_dropout_values:
    print(f"\nTesting LSTM lstm_dropout: {lstm_dropout}")
    model = build_lstm_model_tuned(units=int(best_lstm_units), dropout_rate=lstm_dropout, learning_rate=0.001)
    model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )
    accuracy, precision, recall, f1 = evaluate_lstm_model(model)
    lstm_dropout_results.append({"Units": int(best_lstm_units), "Dropout Rate": lstm_dropout, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
lstm_dropout_df = pd.DataFrame(lstm_dropout_results)
display(lstm_dropout_df)
best_lstm_dropout = lstm_dropout_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Dropout Rate"]
print("Best LSTM Dropout Rate:", best_lstm_dropout)

### LSTM Hyperparameter Tuning: Learning Rate

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

lstm_lr_values = [0.01, 0.001, 0.0001, 0.0005]
lstm_lr_results = []
for lstm_lr in lstm_lr_values:
    print(f"\nTesting LSTM lstm_lr: {lstm_lr}")
    model = build_lstm_model_tuned(units=int(best_lstm_units), dropout_rate=float(best_lstm_dropout), learning_rate=lstm_lr)
    model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )
    accuracy, precision, recall, f1 = evaluate_lstm_model(model)
    lstm_lr_results.append({"Units": int(best_lstm_units), "Dropout Rate": float(best_lstm_dropout), "Learning Rate": lstm_lr, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
lstm_lr_df = pd.DataFrame(lstm_lr_results)
display(lstm_lr_df)
best_lstm_lr = lstm_lr_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Learning Rate"]
print("Best LSTM Learning Rate:", best_lstm_lr)

### LSTM Hyperparameter Tuning: Epochs

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

lstm_epochs_values = [30, 50, 100]
lstm_epoch_results = []
for lstm_epochs in lstm_epochs_values:
    print(f"\nTesting LSTM lstm_epochs: {lstm_epochs}")
    model = build_lstm_model_tuned(units=int(best_lstm_units), dropout_rate=float(best_lstm_dropout), learning_rate=float(best_lstm_lr))
    model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=lstm_epochs,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )
    accuracy, precision, recall, f1 = evaluate_lstm_model(model)
    lstm_epoch_results.append({"Units": int(best_lstm_units), "Dropout Rate": float(best_lstm_dropout), "Learning Rate": float(best_lstm_lr), "Epochs": lstm_epochs, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
lstm_epoch_df = pd.DataFrame(lstm_epoch_results)
display(lstm_epoch_df)
best_lstm_epochs = lstm_epoch_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Epochs"]
print("Best LSTM Epochs:", best_lstm_epochs)

### LSTM Hyperparameter Tuning: Batch Size

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

lstm_batch_values = [32, 64, 128]
lstm_batch_results = []
for lstm_batch in lstm_batch_values:
    print(f"\nTesting LSTM lstm_batch: {lstm_batch}")
    model = build_lstm_model_tuned(units=int(best_lstm_units), dropout_rate=float(best_lstm_dropout), learning_rate=float(best_lstm_lr))
    model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=int(best_lstm_epochs),
        batch_size=lstm_batch,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )
    accuracy, precision, recall, f1 = evaluate_lstm_model(model)
    lstm_batch_results.append({"Units": int(best_lstm_units), "Dropout Rate": float(best_lstm_dropout), "Learning Rate": float(best_lstm_lr), "Epochs": int(best_lstm_epochs), "Batch Size": lstm_batch, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
lstm_batch_df = pd.DataFrame(lstm_batch_results)
display(lstm_batch_df)
best_lstm_batch = lstm_batch_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Batch Size"]
print("Best LSTM Batch Size:", best_lstm_batch)

In [ ]:
best_lstm_params = {
    "Best Units": int(best_lstm_units),
    "Best Dropout Rate": float(best_lstm_dropout),
    "Best Learning Rate": float(best_lstm_lr),
    "Best Epochs": int(best_lstm_epochs),
    "Best Batch Size": int(best_lstm_batch)
}

best_lstm_params

### Final LSTM Performance

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

final_lstm_model = build_lstm_model_tuned(units=best_lstm_params["Best Units"], dropout_rate=best_lstm_params["Best Dropout Rate"], learning_rate=best_lstm_params["Best Learning Rate"])

early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-5)

final_lstm_history = final_lstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=best_lstm_params["Best Epochs"],
    batch_size=best_lstm_params["Best Batch Size"],
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weight,
    verbose=1
)

In [ ]:
y_pred_prob = final_lstm_model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype("int32")

print("Final LSTM Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

final_lstm_accuracy = accuracy_score(y_test, y_pred)
final_lstm_precision = precision_score(y_test, y_pred, zero_division=0)
final_lstm_recall = recall_score(y_test, y_pred, zero_division=0)
final_lstm_f1 = f1_score(y_test, y_pred, zero_division=0)

final_lstm_results = pd.DataFrame([{
    "Model": "LSTM",
    "Accuracy": final_lstm_accuracy,
    "Precision": final_lstm_precision,
    "Recall": final_lstm_recall,
    "F1-Score": final_lstm_f1
}])
display(final_lstm_results)

### Training History of Final LSTM Model: Accuracy and Loss over Epochs

In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(final_lstm_history.history['accuracy'], label='Training Accuracy')
plt.plot(final_lstm_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Final LSTM Model Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(final_lstm_history.history['loss'], label='Training Loss')
plt.plot(final_lstm_history.history['val_loss'], label='Validation Loss')
plt.title('Final LSTM Model Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## BiLSTM Hyperparameter Tuning

In [ ]:
from tensorflow.keras.layers import Bidirectional, SpatialDropout1D
from tensorflow.keras.regularizers import l2

def build_bilstm_model_tuned(units=64, dropout_rate=0.3, learning_rate=0.001, embedding_dim=64):
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
        SpatialDropout1D(0.3),
        Bidirectional(LSTM(units, dropout=dropout_rate)),
        Dense(32, activation='relu', kernel_regularizer=l2(1e-4)),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=learning_rate),
        metrics=['accuracy']
    )
    return model

In [ ]:
def evaluate_bilstm_model(model):
    y_pred_prob = model.predict(X_val_pad)
    y_pred = (y_pred_prob > 0.5).astype("int32")

    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    return accuracy, precision, recall, f1

### BiLSTM Hyperparameter Tuning: Units

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

bilstm_units_values = [64, 128, 256]
bilstm_unit_results = []
for bilstm_units in bilstm_units_values:
    print(f"\nTesting BiLSTM bilstm_units: {bilstm_units}")
    model = build_bilstm_model_tuned(units=bilstm_units, dropout_rate=0.3, learning_rate=0.001)
    model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )
    accuracy, precision, recall, f1 = evaluate_bilstm_model(model)
    bilstm_unit_results.append({"Units": bilstm_units, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
bilstm_unit_df = pd.DataFrame(bilstm_unit_results)
display(bilstm_unit_df)
best_bilstm_units = bilstm_unit_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Units"]
print("Best BiLSTM Units:", best_bilstm_units)

### BiLSTM Hyperparameter Tuning: Dropout Rate

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

bilstm_dropout_values = [0.2, 0.3, 0.5]
bilstm_dropout_results = []
for bilstm_dropout in bilstm_dropout_values:
    print(f"\nTesting BiLSTM bilstm_dropout: {bilstm_dropout}")
    model = build_bilstm_model_tuned(units=int(best_bilstm_units), dropout_rate=bilstm_dropout, learning_rate=0.001)
    model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )
    accuracy, precision, recall, f1 = evaluate_bilstm_model(model)
    bilstm_dropout_results.append({"Units": int(best_bilstm_units), "Dropout Rate": bilstm_dropout, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
bilstm_dropout_df = pd.DataFrame(bilstm_dropout_results)
display(bilstm_dropout_df)
best_bilstm_dropout = bilstm_dropout_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Dropout Rate"]
print("Best BiLSTM Dropout Rate:", best_bilstm_dropout)

### BiLSTM Hyperparameter Tuning: Learning Rate

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

bilstm_lr_values = [0.01, 0.001, 0.0001, 0.0005]
bilstm_lr_results = []
for bilstm_lr in bilstm_lr_values:
    print(f"\nTesting BiLSTM bilstm_lr: {bilstm_lr}")
    model = build_bilstm_model_tuned(units=int(best_bilstm_units), dropout_rate=float(best_bilstm_dropout), learning_rate=bilstm_lr)
    model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )
    accuracy, precision, recall, f1 = evaluate_bilstm_model(model)
    bilstm_lr_results.append({"Units": int(best_bilstm_units), "Dropout Rate": float(best_bilstm_dropout), "Learning Rate": bilstm_lr, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
bilstm_lr_df = pd.DataFrame(bilstm_lr_results)
display(bilstm_lr_df)
best_bilstm_lr = bilstm_lr_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Learning Rate"]
print("Best BiLSTM Learning Rate:", best_bilstm_lr)

### BiLSTM Hyperparameter Tuning: Epochs

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

bilstm_epochs_values = [30, 50, 100]
bilstm_epoch_results = []
for bilstm_epochs in bilstm_epochs_values:
    print(f"\nTesting BiLSTM bilstm_epochs: {bilstm_epochs}")
    model = build_bilstm_model_tuned(units=int(best_bilstm_units), dropout_rate=float(best_bilstm_dropout), learning_rate=float(best_bilstm_lr))
    model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=bilstm_epochs,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )
    accuracy, precision, recall, f1 = evaluate_bilstm_model(model)
    bilstm_epoch_results.append({"Units": int(best_bilstm_units), "Dropout Rate": float(best_bilstm_dropout), "Learning Rate": float(best_bilstm_lr), "Epochs": bilstm_epochs, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
bilstm_epoch_df = pd.DataFrame(bilstm_epoch_results)
display(bilstm_epoch_df)
best_bilstm_epochs = bilstm_epoch_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Epochs"]
print("Best BiLSTM Epochs:", best_bilstm_epochs)

### BiLSTM Hyperparameter Tuning: Batch Size

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

bilstm_batch_values = [32, 64, 128]
bilstm_batch_results = []
for bilstm_batch in bilstm_batch_values:
    print(f"\nTesting BiLSTM bilstm_batch: {bilstm_batch}")
    model = build_bilstm_model_tuned(units=int(best_bilstm_units), dropout_rate=float(best_bilstm_dropout), learning_rate=float(best_bilstm_lr))
    model.fit(
        X_train_pad,
        y_train,
        validation_data=(X_val_pad, y_val),
        epochs=int(best_bilstm_epochs),
        batch_size=bilstm_batch,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        class_weight=class_weight,
        verbose=1
    )
    accuracy, precision, recall, f1 = evaluate_bilstm_model(model)
    bilstm_batch_results.append({"Units": int(best_bilstm_units), "Dropout Rate": float(best_bilstm_dropout), "Learning Rate": float(best_bilstm_lr), "Epochs": int(best_bilstm_epochs), "Batch Size": bilstm_batch, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
bilstm_batch_df = pd.DataFrame(bilstm_batch_results)
display(bilstm_batch_df)
best_bilstm_batch = bilstm_batch_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Batch Size"]
print("Best BiLSTM Batch Size:", best_bilstm_batch)

In [ ]:
best_bilstm_params = {
    "Best Units": int(best_bilstm_units),
    "Best Dropout Rate": float(best_bilstm_dropout),
    "Best Learning Rate": float(best_bilstm_lr),
    "Best Epochs": int(best_bilstm_epochs),
    "Best Batch Size": int(best_bilstm_batch)
}

best_bilstm_params

### Final BiLSTM Performance

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

final_bilstm_model = build_bilstm_model_tuned(units=best_bilstm_params["Best Units"], dropout_rate=best_bilstm_params["Best Dropout Rate"], learning_rate=best_bilstm_params["Best Learning Rate"])

early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-5)

final_bilstm_history = final_bilstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=best_bilstm_params["Best Epochs"],
    batch_size=best_bilstm_params["Best Batch Size"],
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weight,
    verbose=1
)

In [ ]:
y_pred_prob = final_bilstm_model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype("int32")

print("Final BiLSTM Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

final_bilstm_accuracy = accuracy_score(y_test, y_pred)
final_bilstm_precision = precision_score(y_test, y_pred, zero_division=0)
final_bilstm_recall = recall_score(y_test, y_pred, zero_division=0)
final_bilstm_f1 = f1_score(y_test, y_pred, zero_division=0)

final_bilstm_results = pd.DataFrame([{
    "Model": "BiLSTM",
    "Accuracy": final_bilstm_accuracy,
    "Precision": final_bilstm_precision,
    "Recall": final_bilstm_recall,
    "F1-Score": final_bilstm_f1
}])
display(final_bilstm_results)

### Training History of Final BiLSTM Model: Accuracy and Loss over Epochs

In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(final_bilstm_history.history['accuracy'], label='Training Accuracy')
plt.plot(final_bilstm_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Final BiLSTM Model Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(final_bilstm_history.history['loss'], label='Training Loss')
plt.plot(final_bilstm_history.history['val_loss'], label='Validation Loss')
plt.title('Final BiLSTM Model Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Final Model Performance Comparison (CNN vs LSTM vs BiLSTM)

In [ ]:
model_comparison_results = pd.DataFrame([
    {"Model": "CNN", "Accuracy": final_cnn_accuracy, "Precision": final_cnn_precision, "Recall": final_cnn_recall, "F1-Score": final_cnn_f1},
    {"Model": "LSTM", "Accuracy": final_lstm_accuracy, "Precision": final_lstm_precision, "Recall": final_lstm_recall, "F1-Score": final_lstm_f1},
    {"Model": "BiLSTM", "Accuracy": final_bilstm_accuracy, "Precision": final_bilstm_precision, "Recall": final_bilstm_recall, "F1-Score": final_bilstm_f1}
])
display(model_comparison_results)

model_comparison_results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(10, 6))
plt.title("Final Model Performance Comparison (CNN vs LSTM vs BiLSTM)")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Confusion Matrices for Final Models
We will now visualize the confusion matrices for the optimized versions of our CNN, LSTM, and BiLSTM models on the test set.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

final_models = {
    "Final CNN": final_cnn_model,
    "Final LSTM": final_lstm_model,
    "Final BiLSTM": final_bilstm_model
}

plt.figure(figsize=(18, 5))

for i, (name, model) in enumerate(final_models.items()):
    # Predict
    y_probs = model.predict(X_test_pad)
    y_preds = (y_probs > 0.5).astype("int32")

    # Compute matrix
    cm = confusion_matrix(y_test, y_preds)

    # Plot
    plt.subplot(1, 3, i+1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks([0.5, 1.5], ['Not Bully', 'Bully'])
    plt.yticks([0.5, 1.5], ['Not Bully', 'Bully'])

plt.tight_layout()
plt.show()

## BERT Fine-Tuning (Deep Learning Model)

In [ ]:
!pip install -q transformers

In [ ]:
# PyTorch BERT setup (no tf-keras needed)
import os, re
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configuration
model_name = "bert-base-uncased"
bert_max_len = 128
bert_batch_size = 16
bert_epochs = 4
bert_lr = 2e-5
print("Device:", device)

In [ ]:
def clean_text_bert(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    return text.strip()

X_bert = df["Text"].astype(str).apply(clean_text_bert)
y_bert = df["oh_label"]

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_bert, y_bert, test_size=0.2, random_state=42, stratify=y_bert)

Xb_train, Xb_val, yb_train, yb_val = train_test_split(
    Xb_train, yb_train, test_size=0.2, random_state=42, stratify=yb_train)

print("BERT train:", Xb_train.shape)
print("BERT val:", Xb_val.shape)
print("BERT test:", Xb_test.shape)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def encode(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=bert_max_len,
        return_tensors="pt")

def make_loader(texts, labels, shuffle):
    enc = encode(texts)
    ds = TensorDataset(
        enc["input_ids"],
        enc["attention_mask"],
        torch.tensor(labels.values, dtype=torch.long))
    return DataLoader(ds, batch_size=bert_batch_size, shuffle=shuffle)

train_loader = make_loader(Xb_train, yb_train, True)
val_loader   = make_loader(Xb_val,   yb_val,   False)
test_loader  = make_loader(Xb_test,  yb_test,  False)

In [ ]:
# Initialize BERT model (PyTorch)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2).to(device)

bert_cw = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=yb_train)
class_weights = torch.tensor(bert_cw, dtype=torch.float).to(device)

optimizer = torch.optim.AdamW(bert_model.parameters(), lr=bert_lr)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
print("BERT model initialized.")

In [ ]:
def run_epoch(loader, train):
    bert_model.train() if train else bert_model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for input_ids, attn, labels in loader:
            input_ids, attn, labels = input_ids.to(device), attn.to(device), labels.to(device)
            logits = bert_model(input_ids=input_ids, attention_mask=attn).logits
            loss = loss_fn(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            n += labels.size(0)
    return total_loss / n, correct / n

bert_history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}
best_val, patience, wait, best_state = float("inf"), 1, 0, None

print("Training BERT...")
for epoch in range(bert_epochs):
    tr_loss, tr_acc = run_epoch(train_loader, True)
    vl_loss, vl_acc = run_epoch(val_loader, False)
    bert_history["loss"].append(tr_loss);       bert_history["accuracy"].append(tr_acc)
    bert_history["val_loss"].append(vl_loss);   bert_history["val_accuracy"].append(vl_acc)
    print(f"Epoch {epoch+1}/{bert_epochs} - loss {tr_loss:.4f} acc {tr_acc:.4f} - val_loss {vl_loss:.4f} val_acc {vl_acc:.4f}")
    if vl_loss < best_val:
        best_val, wait = vl_loss, 0
        best_state = {k: v.cpu().clone() for k, v in bert_model.state_dict().items()}
    else:
        wait += 1
        if wait > patience:
            print("Early stopping (restoring best weights).")
            break
if best_state is not None:
    bert_model.load_state_dict(best_state)

In [ ]:
# Predict and Evaluate
bert_model.eval()
all_preds = []
with torch.no_grad():
    for input_ids, attn, labels in test_loader:
        input_ids, attn = input_ids.to(device), attn.to(device)
        logits = bert_model(input_ids=input_ids, attention_mask=attn).logits
        all_preds.append(logits.argmax(1).cpu().numpy())
bert_pred = np.concatenate(all_preds)

print("BERT Classification Report:")
print(classification_report(yb_test, bert_pred, zero_division=0))

bert_results = pd.DataFrame([{
    "Model": "BERT",
    "Accuracy": accuracy_score(yb_test, bert_pred),
    "Precision": precision_score(yb_test, bert_pred, zero_division=0),
    "Recall": recall_score(yb_test, bert_pred, zero_division=0),
    "F1-Score": f1_score(yb_test, bert_pred, zero_division=0)
}])
display(bert_results)

In [ ]:
import matplotlib.pyplot as plt

acc = bert_history["accuracy"]
val_acc = bert_history["val_accuracy"]
loss = bert_history["loss"]
val_loss = bert_history["val_loss"]
ep = range(len(acc))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(ep, acc, label="Training Accuracy")
plt.plot(ep, val_acc, label="Validation Accuracy")
plt.title("BERT Fine-Tuned Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(ep, loss, label="Training Loss")
plt.plot(ep, val_loss, label="Validation Loss")
plt.title("BERT Fine-Tuned Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Final Model Performance Comparison (CNN vs LSTM vs BiLSTM vs BERT)

In [ ]:
import matplotlib.pyplot as plt

# Consolidated Final Comparison
model_comparison_all = pd.DataFrame([
    {"Model": "CNN", "Accuracy": final_cnn_accuracy, "Precision": final_cnn_precision, "Recall": final_cnn_recall, "F1-Score": final_cnn_f1},
    {"Model": "LSTM", "Accuracy": final_lstm_accuracy, "Precision": final_lstm_precision, "Recall": final_lstm_recall, "F1-Score": final_lstm_f1},
    {"Model": "BiLSTM", "Accuracy": final_bilstm_accuracy, "Precision": final_bilstm_precision, "Recall": final_bilstm_recall, "F1-Score": final_bilstm_f1},
    {"Model": "BERT", "Accuracy": float(bert_results.iloc[0]["Accuracy"]), "Precision": float(bert_results.iloc[0]["Precision"]), "Recall": float(bert_results.iloc[0]["Recall"]), "F1-Score": float(bert_results.iloc[0]["F1-Score"])}
])

display(model_comparison_all)

model_comparison_all.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(12, 6))
plt.title("Final Cyberbullying Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()